<a href="https://colab.research.google.com/github/TracyK10/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks

*Do staleness and volume correlate with our proxy decline label? Group by bucket, show `n` and declining rate, then verdict.*

In [1]:
import os
from pathlib import Path

import pandas as pd

# Colab upload or local repo path
if Path("/content/content_refresh_anonymized.csv").exists():
    csv_path = "/content/content_refresh_anonymized.csv"
elif Path("data/raw/content_refresh_anonymized.csv").exists():
    csv_path = "data/raw/content_refresh_anonymized.csv"
elif Path("../../data/raw/content_refresh_anonymized.csv").exists():
    os.chdir("../..")
    csv_path = "data/raw/content_refresh_anonymized.csv"
else:
    raise FileNotFoundError(
        "Starter CSV not found. Upload content_refresh_anonymized.csv to /content/ "
        "or run from the repo root."
    )

df = pd.read_csv(csv_path)
df = df[df["impressions_90d"] > 0].copy()
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Signal 1: Staleness (content_age_days)
age_bins = [-1, 90, 180, 365, float("inf")]
age_labels = ["<90", "90-180", "180-365", "365+"]
df["age_bucket"] = pd.cut(df["content_age_days"], bins=age_bins, labels=age_labels)
staleness_table = (
    df.groupby("age_bucket", observed=True)
    .agg(n=("is_declining", "size"), declining_rate=("is_declining", "mean"))
    .reset_index()
)
print("Signal 1 — Staleness (content_age_days)")
display(staleness_table)

# Signal 2: Volume (impressions_90d)
vol_bins = [0, 300, 3000, float("inf")]
vol_labels = ["low", "medium", "high"]
df["volume_bucket"] = pd.cut(df["impressions_90d"], bins=vol_bins, labels=vol_labels)
volume_table = (
    df.groupby("volume_bucket", observed=True)
    .agg(n=("is_declining", "size"), declining_rate=("is_declining", "mean"))
    .reset_index()
)
print("Signal 2 — Volume (impressions_90d)")
display(volume_table)

Signal 1 — Staleness (content_age_days)


,age_bucket,n,declining_rate
0,<90,492,0.668699
1,90-180,11780,0.625552
2,180-365,11368,0.514866
3,365+,6360,0.426258


Signal 2 — Volume (impressions_90d)


,volume_bucket,n,declining_rate
0,low,11256,0.454069
1,medium,10461,0.614664
2,high,8283,0.569963


**Signal 1 — Staleness verdict: OPPOSITE**

Older age buckets show a *lower* declining rate (42.6% at 365+ vs 66.9% at <90), so content age alone does not move in the same direction as our proxy decline label — younger pages are more often flagged `down` in this slice.

**Signal 2 — Volume verdict: MIXED**

Declining rate rises from low-volume (45.4%) to medium/high (61.5% / 57.0%), so visibility matters, but the pattern is not perfectly monotonic — medium volume pages show the highest decline rate, not the highest bucket.

## 2. Encode ONE rule and output queue

*Score = 1 when stale AND visible. Attach reason code and action label, rank, export to `work/outputs/`.*

In [2]:
flagged = (df["content_age_days"] >= 180) & (df["impressions_90d"] >= 500)

queue = df.copy()
queue["baseline_score"] = flagged.astype(int)
queue["reason_code"] = flagged.map({True: "stale_visible_page", False: "none"})
queue["action_label"] = flagged.map({True: "review_for_refresh", False: "none"})

queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False],
).reset_index(drop=True)

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows to {out_path.resolve()}")
print(f"Flagged for review: {flagged.sum():,} pages")

Wrote 30,000 rows to /content/work/outputs/baseline_action_score.csv
Flagged for review: 9,929 pages


## 3. Top-10 review

*Show the top 10 ranked rows, then review each: action, why, and what would make it wrong.*

In [3]:
review_cols = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "content_age_days",
    "impressions_90d",
]
queue[review_cols].head(10)

,content_id,baseline_score,reason_code,action_label,content_age_days,impressions_90d
0,content_5fe46e04994d,1,stale_visible_page,review_for_refresh,537,517715
1,content_aaef01a50def,1,stale_visible_page,review_for_refresh,445,517109
2,content_8c19996aa890,1,stale_visible_page,review_for_refresh,445,509252
3,content_4c36c775b818,1,stale_visible_page,review_for_refresh,445,463103
4,content_2dba2b1f9536,1,stale_visible_page,review_for_refresh,299,443434
5,content_1a9e894be2e2,1,stale_visible_page,review_for_refresh,482,416180
6,content_2c2606c5d176,1,stale_visible_page,review_for_refresh,362,347399
7,content_db5989a78dd3,1,stale_visible_page,review_for_refresh,445,345111
8,content_9532f197bbc8,1,stale_visible_page,review_for_refresh,445,309192
9,content_8e7ba84a972b,1,stale_visible_page,review_for_refresh,224,288426


1. **Action:** review_for_refresh. **Why:** 537 days old with 517,715 impressions — stale and highly visible. **Wrong if:** Traffic is stable year-over-year and the `down` label reflects a short seasonal blip.
2. **Action:** review_for_refresh. **Why:** 445 days old, 517,109 impressions. **Wrong if:** A sibling page absorbed demand (consolidation), not true decay on this URL.
3. **Action:** review_for_refresh. **Why:** 445 days old, 509,252 impressions. **Wrong if:** The decline proxy captures a one-month impression dip that already recovered.
4. **Action:** review_for_refresh. **Why:** 445 days old, 463,103 impressions. **Wrong if:** Client-specific seasonality explains the trend bucket, not content quality.
5. **Action:** review_for_refresh. **Why:** 299 days old, 443,434 impressions. **Wrong if:** High visibility is legacy traffic; updating the page would not shift search demand.
6. **Action:** review_for_refresh. **Why:** 482 days old, 416,180 impressions. **Wrong if:** Position and CTR are stable — only impressions wobbled in the comparison window.
7. **Action:** review_for_refresh. **Why:** 362 days old, 347,399 impressions. **Wrong if:** The page is intentionally deprioritized and traffic loss is expected.
8. **Action:** review_for_refresh. **Why:** 445 days old, 345,111 impressions. **Wrong if:** Recent edits are queued and the stale signal will clear without intervention.
9. **Action:** review_for_refresh. **Why:** 445 days old, 309,192 impressions. **Wrong if:** The `down` label is noise from low click volume despite strong impressions.
10. **Action:** review_for_refresh. **Why:** 224 days old, 288,426 impressions — youngest in the top 10 but still above both cutoffs. **Wrong if:** A rigid 180-day cliff over-ranks age while missing faster-declining younger pages.

## 4. Weak picks

*What does a brittle baseline miss? Where do arbitrary cutoffs fail?*

A **weak pick** for this baseline is any page that passes the rule (`age >= 180` and `impressions >= 500`) but is not truly in decline — for example, a highly visible page whose `trend_direction` is `stable` or `up`, flagged only because the rule ignores trend entirely. The signal audit showed that older age buckets actually have *lower* decline rates in this slice, yet our rule treats age as the primary staleness signal. That mismatch is a weak-pick risk.

The baseline is also brittle at the cutoffs: a page at 179 days with rapidly falling impressions scores 0, while a page at 180 days with 501 impressions scores 1 — arbitrary cliffs that a continuous model can smooth over. This rule uses observable signals only (no product flags, no future-window leakage), but rigidity is its main weakness — which is exactly what ML-08 must beat.

## Self-check

**ML-07 assignment requirements:**

- [x] **Two signal checks** printed with `n` and declining_rate (staleness + volume buckets)
- [x] **Verdicts** under each signal — OPPOSITE (staleness), MIXED (volume)
- [x] **One rule encoded** with `baseline_score`, `reason_code` (`stale_visible_page`), `action_label` (`review_for_refresh`)
- [x] **CSV generated** at `work/outputs/baseline_action_score.csv` (gitignored — stays out of GitHub)
- [x] **Top-10 reviewed** with action, why, and wrong-if caveats for each row

**Submission checklist:**

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.